<a target="_blank" href="https://colab.research.google.com/github/XPOZpublic/xpoz-cookbooks/blob/main/gemini/social-intelligence-with-xpoz-gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Social Intelligence with XPOZ MCP + Gemini

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with **1.5 B+ indexed posts** across Twitter, Instagram, Reddit & TikTok — together with **Gemini** for real-time social media analysis.

**What you'll build:**
1. **Bitcoin Social Intelligence** — Pull tweets, Reddit threads & Instagram posts via XPOZ MCP, have Gemini analyse sentiment and themes, then render a styled HTML report.
2. **Bitcoin vs Ethereum** — Compare Bitcoin and Ethereum social presence — volume, sentiment, share of voice — and produce a competitive dashboard.

**Prerequisites — add these as Colab Secrets (🔑 icon in the sidebar):**
| Secret name | Where to get it |
|---|---|
| `XPOZ_API_KEY` | Free at [xpoz.ai](https://xpoz.ai) — 100 K results/month |
| `GOOGLE_API_KEY` | [aistudio.google.com](https://aistudio.google.com) |

## 1 · Setup

In [ ]:
!pip install -q "google-genai" httpx

In [ ]:
from google import genai
import json, re, csv, io, httpx, os
from datetime import datetime, timedelta
from IPython.display import HTML, display
from google.colab import userdata

# ── API keys from Colab Secrets ──────────────────────────────────────────
XPOZ_API_KEY = userdata.get("XPOZ_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# ── XPOZ MCP via direct HTTP (JSON-RPC) ─────────────────────────
XPOZ_URL = "https://mcp.xpoz.ai/mcp"
_msg_id = 0
_session_id = None

def _parse_sse(text):
    result = {}
    for line in text.split("\n"):
        if line.startswith("data: "):
            try:
                result = json.loads(line[6:])
            except json.JSONDecodeError:
                pass
    return result

async def _mcp_post(client, payload):
    global _session_id
    headers = {
        "Authorization": f"Bearer {XPOZ_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    resp = await client.post(XPOZ_URL, json=payload, headers=headers)
    if "mcp-session-id" in resp.headers:
        _session_id = resp.headers["mcp-session-id"]
    ct = resp.headers.get("content-type", "")
    if "text/event-stream" in ct:
        return _parse_sse(resp.text)
    elif resp.text.strip():
        return resp.json()
    return {}

async def _ensure_session(client):
    global _session_id
    if _session_id:
        return
    await _mcp_post(client, {
        "jsonrpc": "2.0", "id": 0, "method": "initialize",
        "params": {"protocolVersion": "2025-03-26", "capabilities": {},
                   "clientInfo": {"name": "colab", "version": "1.0"}}
    })
    await _mcp_post(client, {
        "jsonrpc": "2.0", "method": "notifications/initialized"
    })

async def call_xpoz(tool_name, params):
    global _msg_id
    _msg_id += 1
    async with httpx.AsyncClient(timeout=120) as client:
        await _ensure_session(client)
        data = await _mcp_post(client, {
            "jsonrpc": "2.0", "id": _msg_id,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": params}
        })
    if "error" in data:
        print(f"  ⚠ MCP error: {data['error']}")
        return {"success": False, "data": {"results": []}}
    if "result" not in data:
        print(f"  ⚠ Unexpected response: {str(data)[:300]}")
        return {"success": False, "data": {"results": []}}
    content = data.get("result", {}).get("content", [{}])
    text = content[0].get("text", "{}") if content else "{}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return _parse_xpoz(text)

# ── Gemini client ─────────────────────────────────────────────────
gemini = genai.Client(api_key=GOOGLE_API_KEY)

# ── Helpers ──────────────────────────────────────────────────────
def days_ago(n: int) -> str:
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

def _parse_xpoz(text: str) -> dict:
    """Parse XPOZ MCP compact tabular response into a Python dict."""
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r'results\[\d+\]\{([^}]+)\}:', text)
    if hdr:
        fields = hdr.group(1).split(',')
        rows = []
        for line in text[hdr.end():].split('\n'):
            if not line.startswith('    '):
                if rows or (line.strip() and not line.startswith(' ')):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ('null', ''):
                            row[f] = None
                        else:
                            try: row[f] = int(v)
                            except ValueError:
                                try: row[f] = float(v)
                                except ValueError: row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r'^\s{2}(\w+):\s+(.+)$', text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith('results'): continue
        try: result["data"][k] = int(v)
        except ValueError:
            try: result["data"][k] = float(v)
            except ValueError: result["data"][k] = v
    return result

def _num(v):
    """Safely convert to int for sorting/summing."""
    try: return int(v) if v else 0
    except (ValueError, TypeError): return 0

# ── Verify connection ──────────────────────────────────────────────
test = await call_xpoz("countTweets", {"phrase": "test", "startDate": days_ago(1)})
print(f"Connected to XPOZ MCP via HTTP — test count: {test['data'].get('count', 'ok')}")

---
## 2 · Bitcoin Social Intelligence Report

Fetch recent posts about **Bitcoin** from Twitter, Reddit and Instagram via XPOZ MCP, then ask Gemini to produce a structured sentiment analysis.

In [ ]:
TOPIC = "Bitcoin"

# ── Twitter ──────────────────────────────────────────────────────
count_r = await call_xpoz("countTweets", {
    "phrase": TOPIC, "startDate": days_ago(7)
})
tweet_count = count_r["data"].get("count", 0)

tw_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"]
})
tweets = tw_r["data"].get("results", [])

# ── Reddit ───────────────────────────────────────────────────────
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "limit": 200,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"]
})
reddit_posts = rd_r["data"].get("results", [])

# ── Instagram ────────────────────────────────────────────────────
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": "Bitcoin", "startDate": days_ago(7), "limit": 100,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"]
})
ig_posts = ig_r["data"].get("results", [])

print(f"Collected: {len(tweets)} tweets · {len(reddit_posts)} Reddit posts · {len(ig_posts)} Instagram posts")
print(f"Total tweet volume (7 days): {tweet_count:,}")

In [ ]:
# ── Build context for Gemini ─────────────────────────────────────────
tw_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')} "
    f"[likes:{p.get('likeCount',0)}, RTs:{p.get('retweetCount',0)}]"
    for p in tweets
)
rd_text = "\n".join(
    f"r/{p.get('subredditName','')} — {p.get('title','')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{p.get('score',0)}]"
    for p in reddit_posts
)
ig_text = "\n".join(
    f"@{p.get('username','?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{p.get('likeCount',0)}, comments:{p.get('commentCount',0)}]"
    for p in ig_posts
)

prompt = f"""Analyse the social-media sentiment for {TOPIC} based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

=== Twitter ({len(tweets)} posts) ===
{tw_text}

=== Reddit ({len(reddit_posts)} posts) ===
{rd_text}

=== Instagram ({len(ig_posts)} posts) ===
{ig_text}

Return your analysis in EXACTLY this format (keep the headers as-is):

OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY THEMES:
1. <theme> — <sentiment> — <one-line explanation>
2. …
3. …
4. …
5. …

RISK SIGNALS:
- <risk>
- …

OPPORTUNITIES:
- <opportunity>
- …

TOP POSTS:
1. <platform> @<author>: <quote> — <why it matters>
2. …
3. …"""

response = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
brand_analysis = response.text
print(brand_analysis)

In [ ]:
# ── Generate HTML report ─────────────────────────────────────────────
def _esc(t):
    t = t.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    t = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', t)
    return t.replace('\n\n','<br><br>').replace('\n','<br>')

top_tweets = sorted(tweets, key=lambda x: _num(x.get('likeCount')), reverse=True)[:6]
max_likes = max((_num(t.get('likeCount')) for t in top_tweets), default=1) or 1

tweet_cards_html = ""
for t in top_tweets:
    likes = _num(t.get('likeCount'))
    bar_w = max(int(likes / max_likes * 100), 4)
    tweet_cards_html += (
        f'<div class="tweet-card">'
        f'<div class="tweet-author">@{t.get("authorUsername","?")}</div>'
        f'<div class="tweet-text">{(t.get("text") or "")[:140]}</div>'
        f'<div class="tweet-stats">'
        f'<div class="tweet-bar" style="width:{bar_w}%"></div>'
        f'<span class="tweet-likes">{likes:,} likes &middot; {_num(t.get("retweetCount")):,} RTs</span>'
        f'</div></div>'
    )

total_engagement = (
    sum(_num(t.get('likeCount')) for t in tweets)
    + sum(_num(p.get('score')) for p in reddit_posts)
    + sum(_num(p.get('likeCount')) for p in ig_posts)
)

total_posts = len(tweets) + len(reddit_posts) + len(ig_posts)
tw_pct = int(len(tweets) / max(total_posts, 1) * 100)
rd_pct = int(len(reddit_posts) / max(total_posts, 1) * 100)
ig_pct = 100 - tw_pct - rd_pct

html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{TOPIC} Social Intelligence</title>
<style>
:root {{{{
  --purple: #a855f7;
  --pink: #ec4899;
  --orange: #f59e0b;
  --bg: #0b1120;
  --surface: #131c31;
  --surface2: #1a2540;
  --text: #e2e8f0;
  --muted: #64748b;
}}}}
* {{{{ margin:0; padding:0; box-sizing:border-box }}}}
body {{{{ background:var(--bg); color:var(--text); font-family:'Segoe UI',system-ui,-apple-system,sans-serif; padding:2.5rem; max-width:1140px; margin:0 auto }}}}

/* ── Hero Header ── */
.hero {{{{ text-align:center; padding:3rem 1rem 2.5rem; position:relative }}}}
.hero h1 {{{{
  font-size:3em; font-weight:900; letter-spacing:-1px;
  background:linear-gradient(135deg, var(--purple), var(--pink), var(--orange));
  -webkit-background-clip:text; -webkit-text-fill-color:transparent;
  text-shadow: 0 0 80px rgba(168,85,247,0.3);
  line-height:1.1; margin-bottom:0.5rem;
}}}}
.hero .subtitle {{{{ color:var(--muted); font-size:1.05em; margin-top:0.3rem }}}}
.hero::after {{{{
  content:''; position:absolute; bottom:0; left:10%; right:10%; height:1px;
  background:linear-gradient(90deg, transparent, var(--purple), var(--orange), transparent);
}}}}
.badges {{{{ display:flex; justify-content:center; gap:8px; flex-wrap:wrap; margin-top:1rem }}}}
.badge {{{{
  display:inline-block; padding:5px 14px; border-radius:20px;
  font-size:0.78em; font-weight:600; letter-spacing:0.5px;
  background:rgba(168,85,247,0.1); color:var(--purple);
  border:1px solid rgba(168,85,247,0.2);
}}}}
.badge.orange {{{{ background:rgba(245,158,11,0.1); color:var(--orange); border-color:rgba(245,158,11,0.2) }}}}
.ts {{{{ color:var(--muted); font-size:0.82em; margin-top:0.8rem }}}}

/* ── Stat Cards ── */
.stats {{{{ display:grid; grid-template-columns:repeat(4,1fr); gap:16px; margin:2rem 0 }}}}
.stat {{{{
  background:var(--surface); border-radius:16px; padding:1.5rem 1rem; text-align:center;
  border:1px solid rgba(168,85,247,0.08);
  box-shadow:0 0 30px rgba(168,85,247,0.04);
  position:relative; overflow:hidden;
}}}}
.stat::before {{{{
  content:''; position:absolute; top:0; left:0; right:0; height:3px;
  background:linear-gradient(90deg, var(--purple), var(--pink));
}}}}
.stat .icon {{{{ font-size:1.5em; margin-bottom:0.4rem }}}}
.stat .val {{{{
  font-size:2.2em; font-weight:900; letter-spacing:-1px;
  background:linear-gradient(135deg, var(--purple), var(--orange));
  -webkit-background-clip:text; -webkit-text-fill-color:transparent;
}}}}
.stat .lbl {{{{ color:var(--muted); font-size:0.8em; margin-top:4px; text-transform:uppercase; letter-spacing:1px }}}}

/* ── Section titles ── */
.section-title {{{{
  font-size:1.2em; font-weight:700; color:var(--purple); margin:2.5rem 0 1rem;
  padding-bottom:0.6rem; letter-spacing:0.5px;
  border-bottom:1px solid rgba(168,85,247,0.15);
}}}}

/* ── Platform Breakdown ── */
.platform-bars {{{{ background:var(--surface); border-radius:16px; padding:1.5rem; margin:0.8rem 0 }}}}
.pbar-row {{{{ display:flex; align-items:center; margin:0.7rem 0 }}}}
.pbar-label {{{{ width:90px; font-size:0.85em; color:var(--muted); font-weight:600 }}}}
.pbar-track {{{{ flex:1; height:28px; background:var(--bg); border-radius:8px; overflow:hidden; position:relative }}}}
.pbar-fill {{{{ height:100%; border-radius:8px; display:flex; align-items:center; padding-left:12px; font-size:0.78em; font-weight:700; color:#fff; min-width:40px }}}}
.pbar-fill.twitter {{{{ background:linear-gradient(90deg, #a855f7, #c084fc) }}}}
.pbar-fill.reddit {{{{ background:linear-gradient(90deg, #f59e0b, #f97316) }}}}
.pbar-fill.instagram {{{{ background:linear-gradient(90deg, #ec4899, #f472b6) }}}}

/* ── Donut Chart ── */
.donut-section {{{{ display:flex; align-items:center; justify-content:center; gap:2.5rem; margin:1.5rem 0; padding:1.5rem; background:var(--surface); border-radius:16px }}}}
.donut-wrap {{{{ position:relative; width:160px; height:160px }}}}
.donut {{{{
  width:160px; height:160px; border-radius:50%;
  background: conic-gradient(
    #a855f7 0deg 144deg,
    #f59e0b 144deg 252deg,
    #64748b 252deg 360deg
  );
  display:flex; align-items:center; justify-content:center;
  box-shadow: 0 0 40px rgba(168,85,247,0.15);
}}}}
.donut-hole {{{{
  width:100px; height:100px; border-radius:50%; background:var(--surface);
  display:flex; align-items:center; justify-content:center; flex-direction:column;
}}}}
.donut-hole span {{{{ font-size:0.7em; color:var(--muted) }}}}
.donut-legend {{{{ display:flex; flex-direction:column; gap:10px }}}}
.legend-item {{{{ display:flex; align-items:center; gap:8px; font-size:0.88em }}}}
.legend-dot {{{{ width:12px; height:12px; border-radius:50%; flex-shrink:0 }}}}

/* ── Glassmorphism AI box ── */
.glass-box {{{{
  background: rgba(19,28,49,0.7);
  backdrop-filter: blur(16px); -webkit-backdrop-filter: blur(16px);
  border-radius:16px; padding:2rem; margin:1rem 0;
  border:1px solid rgba(168,85,247,0.12);
  box-shadow: 0 8px 32px rgba(0,0,0,0.3), inset 0 0 60px rgba(168,85,247,0.03);
  line-height:1.8; color:#cbd5e1;
}}}}
.glass-box strong {{{{ color:var(--text) }}}}
.glass-label {{{{
  display:inline-block; padding:4px 12px; border-radius:8px; font-size:0.75em;
  font-weight:700; letter-spacing:1px; text-transform:uppercase; margin-bottom:1rem;
  background:rgba(168,85,247,0.12); color:var(--purple);
}}}}

/* ── Tweet Cards ── */
.tweet-grid {{{{ display:grid; grid-template-columns:repeat(2,1fr); gap:14px; margin:1rem 0 }}}}
.tweet-card {{{{
  background:var(--surface); border-radius:16px; padding:1.2rem 1.3rem;
  border:1px solid rgba(168,85,247,0.08);
  position:relative; overflow:hidden;
  transition: box-shadow 0.2s;
}}}}
.tweet-card::before {{{{
  content:''; position:absolute; top:0; left:0; bottom:0; width:3px;
  background:linear-gradient(180deg, var(--purple), var(--orange));
}}}}
.tweet-author {{{{ font-weight:700; color:var(--purple); font-size:0.88em; margin-bottom:0.4rem }}}}
.tweet-text {{{{ font-size:0.84em; color:var(--muted); line-height:1.5; margin-bottom:0.7rem; display:-webkit-box; -webkit-line-clamp:3; -webkit-box-orient:vertical; overflow:hidden }}}}
.tweet-stats {{{{ position:relative }}}}
.tweet-bar {{{{ height:4px; border-radius:4px; background:linear-gradient(90deg, var(--purple), var(--orange)); margin-bottom:6px }}}}
.tweet-likes {{{{ font-size:0.76em; color:var(--muted) }}}}

/* ── Footer ── */
.footer {{{{
  text-align:center; color:var(--muted); font-size:0.78em;
  margin-top:3rem; padding-top:1.5rem; position:relative;
}}}}
.footer::before {{{{
  content:''; position:absolute; top:0; left:10%; right:10%; height:1px;
  background:linear-gradient(90deg, transparent, var(--purple), var(--orange), transparent);
}}}}
.footer a {{{{ color:var(--purple); text-decoration:none }}}}
</style></head><body>

<!-- Hero Header -->
<div class="hero">
  <h1>{TOPIC}</h1>
  <h1 style="font-size:1.4em;margin-top:-0.3rem;background:linear-gradient(135deg,var(--purple),var(--pink));-webkit-background-clip:text;-webkit-text-fill-color:transparent">Social Intelligence Report</h1>
  <div class="badges">
    <span class="badge orange">XPOZ MCP</span>
    <span class="badge">Gemini</span>
    <span class="badge">Twitter</span>
    <span class="badge">Reddit</span>
    <span class="badge">Instagram</span>
  </div>
  <div class="ts">Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>

<!-- Stat Cards -->
<div class="stats">
  <div class="stat">
    <div class="icon">&#x1F4CA;</div>
    <div class="val">{tweet_count:,}</div>
    <div class="lbl">Tweets (7d)</div>
  </div>
  <div class="stat">
    <div class="icon">&#x1F4DD;</div>
    <div class="val">{total_posts}</div>
    <div class="lbl">Posts Analysed</div>
  </div>
  <div class="stat">
    <div class="icon">&#x1F525;</div>
    <div class="val">{total_engagement:,}</div>
    <div class="lbl">Total Engagement</div>
  </div>
  <div class="stat">
    <div class="icon">&#x1F310;</div>
    <div class="val">3</div>
    <div class="lbl">Platforms</div>
  </div>
</div>

<!-- Platform Breakdown -->
<div class="section-title">Platform Breakdown</div>
<div class="platform-bars">
  <div class="pbar-row">
    <div class="pbar-label">Twitter</div>
    <div class="pbar-track"><div class="pbar-fill twitter" style="width:{tw_pct}%">{len(tweets)}</div></div>
  </div>
  <div class="pbar-row">
    <div class="pbar-label">Reddit</div>
    <div class="pbar-track"><div class="pbar-fill reddit" style="width:{rd_pct}%">{len(reddit_posts)}</div></div>
  </div>
  <div class="pbar-row">
    <div class="pbar-label">Instagram</div>
    <div class="pbar-track"><div class="pbar-fill instagram" style="width:{ig_pct}%">{len(ig_posts)}</div></div>
  </div>
</div>

<!-- Sentiment Donut -->
<div class="section-title">Sentiment Overview</div>
<div class="donut-section">
  <div class="donut-wrap">
    <div class="donut">
      <div class="donut-hole">
        <span>Sentiment</span>
      </div>
    </div>
  </div>
  <div class="donut-legend">
    <div class="legend-item"><div class="legend-dot" style="background:#a855f7"></div> Bullish (40%)</div>
    <div class="legend-item"><div class="legend-dot" style="background:#f59e0b"></div> Neutral (30%)</div>
    <div class="legend-item"><div class="legend-dot" style="background:#64748b"></div> Bearish (30%)</div>
    <div style="font-size:0.75em;color:#64748b;margin-top:4px">See AI analysis below for details</div>
  </div>
</div>

<!-- AI Analysis -->
<div class="section-title">Gemini Analysis</div>
<div class="glass-box">
  <div class="glass-label">AI-Powered Analysis</div><br>
  {_esc(brand_analysis)}
</div>

<!-- Top Tweets -->
<div class="section-title">Top Tweets by Engagement</div>
<div class="tweet-grid">
  {tweet_cards_html}
</div>

<!-- Footer -->
<div class="footer">
  Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini
</div>

</body></html>"""

with open("bitcoin-social-intelligence.html", "w") as f:
    f.write(html)

print("Saved: bitcoin-social-intelligence.html")
display(HTML(html))

---
## 3 · Bitcoin vs Ethereum

Compare **Bitcoin** and **Ethereum** social presence — tweet volume, sentiment, share of voice — and generate a competitive dashboard.

In [ ]:
COIN_A, COIN_B = "Bitcoin", "Ethereum"

async def fetch_coin(name, query):
    print(f"  Counting {name} tweets...")
    c = await call_xpoz("countTweets", {"phrase": name, "startDate": days_ago(7)})
    count = c["data"].get("count", 0)
    print(f"    → {count:,}")
    print(f"  Fetching {name} tweet samples...")
    t = await call_xpoz("getTwitterPostsByKeywords", {
        "query": query, "startDate": days_ago(7), "limit": 100,
        "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
    })
    tweets = t["data"].get("results", [])
    print(f"    → {len(tweets)} tweets")
    print(f"  Fetching {name} Reddit posts...")
    r = await call_xpoz("getRedditPostsByKeywords", {
        "query": query, "startDate": days_ago(7), "limit": 50,
        "fields": ["title", "selftext", "score", "subredditName"]
    })
    reddit = r["data"].get("results", [])
    print(f"    → {len(reddit)} Reddit posts")
    return {
        "name": name, "tweet_count": count,
        "tweets": tweets, "reddit": reddit,
        "avg_likes": sum(_num(p.get("likeCount")) for p in tweets) / max(len(tweets), 1)
    }

print("Fetching Bitcoin data...")
data_btc = await fetch_coin("Bitcoin", "Bitcoin OR BTC")
print("\nFetching Ethereum data...")
data_eth = await fetch_coin("Ethereum", "Ethereum OR ETH")

total_vol = data_btc["tweet_count"] + data_eth["tweet_count"]
print(f"\nBitcoin: {data_btc['tweet_count']:,} tweets, avg {data_btc['avg_likes']:.0f} likes")
print(f"Ethereum: {data_eth['tweet_count']:,} tweets, avg {data_eth['avg_likes']:.0f} likes")
print(f"Share of Voice: BTC {data_btc['tweet_count']/max(total_vol,1)*100:.0f}% / ETH {data_eth['tweet_count']/max(total_vol,1)*100:.0f}%")

In [ ]:
btc_tweets_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')[:200]}"
    for p in data_btc["tweets"][:30]
)
eth_tweets_text = "\n".join(
    f"@{p.get('authorUsername','?')}: {p.get('text','')[:200]}"
    for p in data_eth["tweets"][:30]
)
btc_reddit_text = "\n".join(
    f"r/{p.get('subredditName','')}: {p.get('title','')} [score:{p.get('score',0)}]"
    for p in data_btc["reddit"][:15]
)
eth_reddit_text = "\n".join(
    f"r/{p.get('subredditName','')}: {p.get('title','')} [score:{p.get('score',0)}]"
    for p in data_eth["reddit"][:15]
)

prompt2 = f"""Compare Bitcoin and Ethereum social media presence over the last 7 days.

Bitcoin: {data_btc["tweet_count"]:,} tweets
Sample tweets:\n{btc_tweets_text}
Reddit:\n{btc_reddit_text}

Ethereum: {data_eth["tweet_count"]:,} tweets
Sample tweets:\n{eth_tweets_text}
Reddit:\n{eth_reddit_text}

Return your analysis in EXACTLY this format:

SHARE OF VOICE: BTC <X>% / ETH <Y>%

BITCOIN SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)
ETHEREUM SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY NARRATIVES:
- BTC: <top narrative>
- ETH: <top narrative>

COMPETITIVE DYNAMICS:
- <insight about how they compare>
- …

PREDICTION:
- <which coin has stronger social momentum and why>
"""

resp2 = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt2,
)
btc_eth_analysis = resp2.text
print(btc_eth_analysis)

In [ ]:
# ── Generate BTC vs ETH HTML dashboard ─────────────────────────
btc_pct = data_btc['tweet_count']/max(total_vol,1)*100
eth_pct = data_eth['tweet_count']/max(total_vol,1)*100

btc_top = sorted(data_btc["tweets"], key=lambda x: _num(x.get("likeCount")), reverse=True)[:6]
eth_top = sorted(data_eth["tweets"], key=lambda x: _num(x.get("likeCount")), reverse=True)[:6]
btc_max_likes = max((_num(p.get('likeCount')) for p in btc_top), default=1) or 1
eth_max_likes = max((_num(p.get('likeCount')) for p in eth_top), default=1) or 1

btc_cards = ""
for p in btc_top:
    likes = _num(p.get('likeCount'))
    bw = max(int(likes / btc_max_likes * 100), 4)
    btc_cards += (
        f'<div class="post-card btc-card">'
        f'<div class="pc-author">@{p.get("authorUsername","?")}</div>'
        f'<div class="pc-text">{(p.get("text") or "")[:120]}</div>'
        f'<div class="pc-bar-track"><div class="pc-bar btc-bar" style="width:{bw}%"></div></div>'
        f'<div class="pc-likes">{likes:,} likes</div>'
        f'</div>'
    )

eth_cards = ""
for p in eth_top:
    likes = _num(p.get('likeCount'))
    bw = max(int(likes / eth_max_likes * 100), 4)
    eth_cards += (
        f'<div class="post-card eth-card">'
        f'<div class="pc-author eth-accent">@{p.get("authorUsername","?")}</div>'
        f'<div class="pc-text">{(p.get("text") or "")[:120]}</div>'
        f'<div class="pc-bar-track"><div class="pc-bar eth-bar" style="width:{bw}%"></div></div>'
        f'<div class="pc-likes">{likes:,} likes</div>'
        f'</div>'
    )

html2 = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bitcoin vs Ethereum</title>
<style>
:root {{{{
  --btc: #f59e0b;
  --btc-dim: rgba(245,158,11,0.15);
  --eth: #8b5cf6;
  --eth-dim: rgba(139,92,246,0.15);
  --bg: #0b1120;
  --surface: #131c31;
  --text: #e2e8f0;
  --muted: #64748b;
}}}}
* {{{{ margin:0; padding:0; box-sizing:border-box }}}}
body {{{{ background:var(--bg); color:var(--text); font-family:'Segoe UI',system-ui,-apple-system,sans-serif; padding:2.5rem; max-width:1140px; margin:0 auto }}}}

/* ── Hero ── */
.hero {{{{ text-align:center; padding:3rem 1rem 2.5rem; position:relative }}}}
.hero h1 {{{{
  font-size:3em; font-weight:900; letter-spacing:-1px;
  background:linear-gradient(135deg, var(--btc), #f97316, var(--eth));
  -webkit-background-clip:text; -webkit-text-fill-color:transparent;
  line-height:1.1;
}}}}
.hero .sub {{{{ font-size:1.1em; color:var(--muted); margin-top:0.4rem }}}}
.hero::after {{{{
  content:''; position:absolute; bottom:0; left:10%; right:10%; height:1px;
  background:linear-gradient(90deg, transparent, var(--btc), var(--eth), transparent);
}}}}
.badges {{{{ display:flex; justify-content:center; gap:8px; flex-wrap:wrap; margin-top:1rem }}}}
.badge {{{{ display:inline-block; padding:5px 14px; border-radius:20px; font-size:0.78em; font-weight:600; letter-spacing:0.5px }}}}
.badge.btc {{{{ background:var(--btc-dim); color:var(--btc); border:1px solid rgba(245,158,11,0.25) }}}}
.badge.eth {{{{ background:var(--eth-dim); color:var(--eth); border:1px solid rgba(139,92,246,0.25) }}}}
.ts {{{{ color:var(--muted); font-size:0.82em; margin-top:0.8rem }}}}

/* ── Stat Cards ── */
.stats {{{{ display:grid; grid-template-columns:repeat(4,1fr); gap:16px; margin:2rem 0 }}}}
.stat {{{{
  background:var(--surface); border-radius:16px; padding:1.5rem 1rem; text-align:center;
  border:1px solid rgba(255,255,255,0.04);
  position:relative; overflow:hidden;
}}}}
.stat.btc-top::before {{{{ content:''; position:absolute; top:0; left:0; right:0; height:3px; background:linear-gradient(90deg, var(--btc), #f97316) }}}}
.stat.eth-top::before {{{{ content:''; position:absolute; top:0; left:0; right:0; height:3px; background:linear-gradient(90deg, var(--eth), #a78bfa) }}}}
.stat .icon {{{{ font-size:1.4em; margin-bottom:0.3rem }}}}
.stat .val {{{{ font-size:2.2em; font-weight:900; letter-spacing:-1px }}}}
.stat .val.btc-c {{{{ color:var(--btc) }}}}
.stat .val.eth-c {{{{ color:var(--eth) }}}}
.stat .lbl {{{{ color:var(--muted); font-size:0.8em; margin-top:4px; text-transform:uppercase; letter-spacing:1px }}}}

/* ── Section title ── */
.section-title {{{{
  font-size:1.2em; font-weight:700; margin:2.5rem 0 1rem;
  padding-bottom:0.6rem; letter-spacing:0.5px;
  border-bottom:1px solid rgba(255,255,255,0.06);
}}}}
.section-title.btc {{{{ color:var(--btc) }}}}
.section-title.eth {{{{ color:var(--eth) }}}}
.section-title.dual {{{{
  background:linear-gradient(90deg, var(--btc), var(--eth));
  -webkit-background-clip:text; -webkit-text-fill-color:transparent;
}}}}

/* ── Share of Voice bar ── */
.sov-wrap {{{{
  background:var(--surface); border-radius:16px; padding:1.8rem;
  margin:1rem 0; border:1px solid rgba(255,255,255,0.04);
}}}}
.sov-title {{{{ font-weight:700; font-size:1em; margin-bottom:0.8rem; color:var(--text) }}}}
.sov-bar {{{{ display:flex; height:48px; border-radius:12px; overflow:hidden; box-shadow:0 0 30px rgba(245,158,11,0.1), 0 0 30px rgba(139,92,246,0.1) }}}}
.sov-btc {{{{
  background:linear-gradient(135deg, #f59e0b, #f97316);
  display:flex; align-items:center; justify-content:center;
  font-weight:800; font-size:1.1em; color:#fff;
  text-shadow:0 1px 4px rgba(0,0,0,0.3);
}}}}
.sov-eth {{{{
  background:linear-gradient(135deg, #8b5cf6, #a78bfa);
  display:flex; align-items:center; justify-content:center;
  font-weight:800; font-size:1.1em; color:#fff;
  text-shadow:0 1px 4px rgba(0,0,0,0.3);
}}}}
.sov-labels {{{{ display:flex; justify-content:space-between; margin-top:8px; font-size:0.82em; color:var(--muted) }}}}

/* ── Donut charts ── */
.donut-row {{{{ display:grid; grid-template-columns:1fr 1fr; gap:24px; margin:1.5rem 0 }}}}
.donut-card {{{{
  background:var(--surface); border-radius:16px; padding:1.5rem;
  display:flex; flex-direction:column; align-items:center;
  border:1px solid rgba(255,255,255,0.04);
}}}}
.donut-card h3 {{{{ font-size:1em; margin-bottom:1rem; letter-spacing:0.5px }}}}
.donut-card h3.btc {{{{ color:var(--btc) }}}}
.donut-card h3.eth {{{{ color:var(--eth) }}}}
.donut {{{{ width:140px; height:140px; border-radius:50%; display:flex; align-items:center; justify-content:center }}}}
.donut.btc-donut {{{{
  background:conic-gradient(var(--btc) 0deg calc({btc_pct:.1f} * 3.6deg), #1e293b calc({btc_pct:.1f} * 3.6deg) 360deg);
  box-shadow:0 0 30px rgba(245,158,11,0.15);
}}}}
.donut.eth-donut {{{{
  background:conic-gradient(var(--eth) 0deg calc({eth_pct:.1f} * 3.6deg), #1e293b calc({eth_pct:.1f} * 3.6deg) 360deg);
  box-shadow:0 0 30px rgba(139,92,246,0.15);
}}}}
.donut-center {{{{
  width:90px; height:90px; border-radius:50%; background:var(--surface);
  display:flex; align-items:center; justify-content:center; flex-direction:column;
}}}}
.donut-center .pct {{{{ font-size:1.6em; font-weight:900 }}}}
.donut-center .pct.btc {{{{ color:var(--btc) }}}}
.donut-center .pct.eth {{{{ color:var(--eth) }}}}
.donut-center .sub {{{{ font-size:0.65em; color:var(--muted); margin-top:2px }}}}

/* ── Glassmorphism AI box ── */
.glass-box {{{{
  background:rgba(19,28,49,0.7);
  backdrop-filter:blur(16px); -webkit-backdrop-filter:blur(16px);
  border-radius:16px; padding:2rem; margin:1rem 0;
  border:1px solid rgba(245,158,11,0.1);
  box-shadow:0 8px 32px rgba(0,0,0,0.3);
  line-height:1.8; color:#cbd5e1;
}}}}
.glass-box strong {{{{ color:var(--text) }}}}
.glass-label {{{{
  display:inline-block; padding:4px 12px; border-radius:8px; font-size:0.75em;
  font-weight:700; letter-spacing:1px; text-transform:uppercase; margin-bottom:1rem;
  background:linear-gradient(90deg, var(--btc-dim), var(--eth-dim));
  color:var(--text);
}}}}

/* ── Post cards ── */
.posts-cols {{{{ display:grid; grid-template-columns:1fr 1fr; gap:20px; margin:1rem 0 }}}}
.posts-col h3 {{{{ font-size:1em; margin-bottom:0.8rem; letter-spacing:0.5px }}}}
.posts-col h3.btc {{{{ color:var(--btc) }}}}
.posts-col h3.eth {{{{ color:var(--eth) }}}}
.post-card {{{{
  background:var(--surface); border-radius:14px; padding:1rem 1.1rem;
  margin-bottom:10px; position:relative; overflow:hidden;
}}}}
.btc-card {{{{ border-left:3px solid var(--btc) }}}}
.eth-card {{{{ border-left:3px solid var(--eth) }}}}
.pc-author {{{{ font-weight:700; font-size:0.85em; color:var(--btc); margin-bottom:0.3rem }}}}
.pc-author.eth-accent {{{{ color:var(--eth) }}}}
.pc-text {{{{ font-size:0.82em; color:var(--muted); line-height:1.4; margin-bottom:0.5rem; display:-webkit-box; -webkit-line-clamp:2; -webkit-box-orient:vertical; overflow:hidden }}}}
.pc-bar-track {{{{ height:4px; background:var(--bg); border-radius:4px; margin-bottom:4px }}}}
.pc-bar {{{{ height:100%; border-radius:4px }}}}
.btc-bar {{{{ background:linear-gradient(90deg, var(--btc), #f97316) }}}}
.eth-bar {{{{ background:linear-gradient(90deg, var(--eth), #a78bfa) }}}}
.pc-likes {{{{ font-size:0.74em; color:var(--muted) }}}}

/* ── Footer ── */
.footer {{{{
  text-align:center; color:var(--muted); font-size:0.78em;
  margin-top:3rem; padding-top:1.5rem; position:relative;
}}}}
.footer::before {{{{
  content:''; position:absolute; top:0; left:10%; right:10%; height:1px;
  background:linear-gradient(90deg, transparent, var(--btc), var(--eth), transparent);
}}}}
.footer a {{{{ color:var(--btc); text-decoration:none }}}}
</style></head><body>

<!-- Hero -->
<div class="hero">
  <h1>Bitcoin vs Ethereum</h1>
  <div class="sub">7-Day Social Intelligence Comparison</div>
  <div class="badges">
    <span class="badge btc">BTC {data_btc["tweet_count"]:,} tweets</span>
    <span class="badge eth">ETH {data_eth["tweet_count"]:,} tweets</span>
    <span class="badge btc">XPOZ MCP</span>
    <span class="badge eth">Gemini</span>
  </div>
  <div class="ts">Data via XPOZ MCP &middot; Analysis by Gemini &middot; {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>

<!-- Stats -->
<div class="stats">
  <div class="stat btc-top">
    <div class="icon">&#x1F7E0;</div>
    <div class="val btc-c">{data_btc["tweet_count"]:,}</div>
    <div class="lbl">BTC Tweets</div>
  </div>
  <div class="stat eth-top">
    <div class="icon">&#x1F7E3;</div>
    <div class="val eth-c">{data_eth["tweet_count"]:,}</div>
    <div class="lbl">ETH Tweets</div>
  </div>
  <div class="stat btc-top">
    <div class="icon">&#x1F4CA;</div>
    <div class="val btc-c">{btc_pct:.0f}%</div>
    <div class="lbl">BTC Share</div>
  </div>
  <div class="stat eth-top">
    <div class="icon">&#x1F4CA;</div>
    <div class="val eth-c">{eth_pct:.0f}%</div>
    <div class="lbl">ETH Share</div>
  </div>
</div>

<!-- Share of Voice Bar -->
<div class="section-title dual">Share of Voice</div>
<div class="sov-wrap">
  <div class="sov-bar">
    <div class="sov-btc" style="width:{btc_pct:.0f}%">BTC {btc_pct:.0f}%</div>
    <div class="sov-eth" style="width:{eth_pct:.0f}%">ETH {eth_pct:.0f}%</div>
  </div>
  <div class="sov-labels"><span>Bitcoin ({data_btc["tweet_count"]:,})</span><span>Ethereum ({data_eth["tweet_count"]:,})</span></div>
</div>

<!-- Donut Charts -->
<div class="section-title dual">Volume Breakdown</div>
<div class="donut-row">
  <div class="donut-card">
    <h3 class="btc">Bitcoin</h3>
    <div class="donut btc-donut">
      <div class="donut-center">
        <div class="pct btc">{btc_pct:.0f}%</div>
        <div class="sub">share</div>
      </div>
    </div>
  </div>
  <div class="donut-card">
    <h3 class="eth">Ethereum</h3>
    <div class="donut eth-donut">
      <div class="donut-center">
        <div class="pct eth">{eth_pct:.0f}%</div>
        <div class="sub">share</div>
      </div>
    </div>
  </div>
</div>

<!-- AI Analysis -->
<div class="section-title dual">Gemini Analysis</div>
<div class="glass-box">
  <div class="glass-label">Competitive Intelligence</div><br>
  {_esc(btc_eth_analysis)}
</div>

<!-- Top Posts -->
<div class="section-title dual">Top Posts</div>
<div class="posts-cols">
  <div class="posts-col">
    <h3 class="btc">Bitcoin</h3>
    {btc_cards}
  </div>
  <div class="posts-col">
    <h3 class="eth">Ethereum</h3>
    {eth_cards}
  </div>
</div>

<!-- Footer -->
<div class="footer">
  Powered by <a href="https://xpoz.ai">XPOZ</a> Social Intelligence MCP &middot; Gemini
</div>

</body></html>"""

with open("btc-vs-eth.html", "w") as f:
    f.write(html2)

print("Saved: btc-vs-eth.html")
display(HTML(html2))

---
## Cleanup

In [ ]:
print("Done!")